### running sql to fetch data

In [ ]:
import pandas as pd
import sys
sys.path.extend(['/home/jovyan/modules'])

from db_connector import *

query = '''

    SELECT 
        film_id,
        COUNT(DISTINCT CASE WHEN event_type = 'visit' THEN user_id END) as unique_visits,
        COUNT(CASE WHEN event_type = 'visit' THEN 1 END) as visits,
        
        COUNT(DISTINCT CASE WHEN event_type = 'click' THEN user_id END) as unique_clickers,
        COUNT(CASE WHEN event_type = 'click' THEN 1 END) as clicks ,
        
        COUNT(DISTINCT CASE WHEN event_type = 'add_to_basket' THEN user_id END) as unique_adds,
        COUNT(CASE WHEN event_type = 'add_to_basket' THEN 1 END) as adds,
        
        COUNT(DISTINCT CASE WHEN event_type = 'payment' THEN user_id END) as unique_payments,
        COUNT(CASE WHEN event_type = 'payment' THEN 1 END) as payments

    FROM vod.events
    WHERE create_date >= CURRENT_DATE - INTERVAL '30 days'
    GROUP BY film_id 
    
'''

event_df = pd.read_sql_query(query,bootcamp_db)
event_df.head()

### cleaning data

In [ ]:
unique_visits = event_df["unique_visits"][0]
total_visits = event_df["visits"][0]
print(unique_visits ,"    ",total_visits)
event_df = event_df[event_df['film_id'] != -1].copy()
event_df = event_df.drop(columns=["visits","unique_visits"]).reset_index(drop=True)
event_df

In [ ]:
smoothing_clicks = 10 
smoothing_adds = 5
smoothing_payment = 2

event_df["click_add_rate"] = (event_df["adds"]+ smoothing_adds) / (event_df["clicks"]+ smoothing_clicks) # fav
event_df["add_pay_rate"] = (event_df["payments"] + smoothing_payment) / (event_df["adds"]+smoothing_adds) # money
event_df

In [ ]:
import numpy as np
score = event_df[["film_id"]].copy()
score["cr_score"] = (event_df["click_add_rate"] * 0.4 + event_df["add_pay_rate"] * 0.6) * np.log1p(event_df["clicks"])

score

In [ ]:
score.sort_values(["cr_score"],ascending = False).head(100)

## Visualization Result

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

#Style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 10)

fig, axes = plt.subplots(3, 1, figsize=(10, 14))

#Scatter Plot
sns.scatterplot(
    data=event_df.assign(final_score=score['cr_score']), 
    x='clicks', 
    y='final_score', 
    hue='add_pay_rate', 
    palette='viridis', 
    ax=axes[0],
    alpha=0.6
)
axes[0].set_title('Relationship: Clicks vs. Final Score\n(Color indicates Conversion Rate)', fontsize=14)
axes[0].set_xlabel('Total Clicks', fontsize=12)
axes[0].set_ylabel('Final Ranking Score', fontsize=12)

#Distribution Plot
sns.histplot(score['cr_score'], kde=True, color='skyblue', ax=axes[1])
axes[1].set_title('Distribution of Final Scores', fontsize=14)
axes[1].set_xlabel('Score', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)

#Box Plot
sns.boxplot(data=df_plot, x="group", y="cr_score", hue="group",palette="viridis", width=0.5, showfliers=False,legend=False ,ax=axes[2])

sns.stripplot(data=df_plot, x="group", y="cr_score", color="black", size=2, alpha=0.3 , ax=axes[2])

axes[2].set_title(f"Statistical Separation: Top {n} Films vs Others", fontsize=15)
axes[2].grid(axis='y', linestyle='--', alpha=0.7)

plt.savefig('cr_result.png')
plt.tight_layout()
plt.show()